# Product Purchase Propensity 
## Data Preparation (real source data, sampled) 

**Directive reference:** Use Case 1 — Product Purchase Propensity (`Alex_Lee_Promotion_Data_Science_Directive`)

Run this notebook with both `connection_to_SANDBOX` (raw source tables) and `inviso_sandbox` (curated output) added as Lakehouses -- neither is the default Lakehouse for this session. Customers, products, and transactions all load from Alex Lee's real source tables, scoped to a sample of `NUM_SAMPLE_HOUSEHOLDS` households with real transaction history (and whatever products they actually bought) so this first real-data pass stays small and fast. Raise or remove that sample limit once the pipeline has been validated end to end.

Seasonality: the directive calls for a seasonal adjustment (calendar month / holiday timing / Alex Lee's breadth-of-spend studies) once the business team has reviewed and defined it. `seasonal_index` below is a placeholder feature (month-of-year, 1-12) standing in for that adjustment until the real seasonal factor is defined.


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import mlflow
import mlflow.spark


StatementMeta(, 1bdd9bd9-7b7e-488c-8793-0185123f39ec, 9, Finished, Available, Finished, False)

In [4]:
# Fabric notebook parameters (mark this cell as a "parameters" cell if used with pipelines)
MODEL_KEY = "product_purchase_propensity"
EXPERIMENT_NAME = "PromotionEngine_ProductPurchasePropensity"

# Raw source tables (loyalty_household, loyalty_card, item_master, line, header) live in the
# data-modelling Lakehouse -- add it to this notebook's Lakehouse explorer.
SOURCE_LAKEHOUSE = "connection_to_SANDBOX"
SOURCE_SCHEMA = "dbo"

def source_table(name):
    return f"{SOURCE_LAKEHOUSE}.{SOURCE_SCHEMA}.{name}"

# Curated/output tables land in a separate Lakehouse from the source tables above --
# also add inviso_sandbox to this notebook's Lakehouse explorer (neither is the default
# Lakehouse for this session).
OUTPUT_LAKEHOUSE = "inviso_sandbox"
OUTPUT_SCHEMA = "dbo"

def output_table(name):
    return f"{OUTPUT_LAKEHOUSE}.{OUTPUT_SCHEMA}.{MODEL_KEY}_{name}"

# Template scale: sample NUM_SAMPLE_HOUSEHOLDS households with real transaction history, then
# keep whatever products they actually bought (no separate product sample/limit) -- this first
# pass stays small and fast without risking a household/product sample that don't overlap.
NUM_SAMPLE_HOUSEHOLDS = 1000

mlflow.set_experiment(EXPERIMENT_NAME)

# Here's the data we still need:

StatementMeta(, 1bdd9bd9-7b7e-488c-8793-0185123f39ec, 10, Finished, Available, Finished, False)

<Experiment: artifact_location='sds://onelakeeastus2.pbidedicated.windows.net/85cb6cdc-bedf-48fd-afeb-7ae771ca5771/19d958a4-aa8d-4ac3-9311-789ea2b11dec', creation_time=1787341661410, experiment_id='19d958a4-aa8d-4ac3-9311-789ea2b11dec', last_update_time=1787341661410, lifecycle_stage='active', name='PromotionEngine_ProductPurchasePropensity', tags={}>

## 1. Source data

Customers, products, and transactions are all read directly from Alex Lee's real Lakehouse tables in `connection_to_SANDBOX.dbo`: `loyalty_household` / `loyalty_card`, `item_master`, and `line` / `header`. This first pass is scoped to a sample of `NUM_SAMPLE_HOUSEHOLDS` households (and whatever products they actually bought) so it stays small and fast to iterate on -- raise or remove that limit once the pipeline is validated. Curated tables are written to `inviso_sandbox.dbo` (a separate Lakehouse from the source data above).

**Sampling strategy:** only households are sampled -- `NUM_SAMPLE_HOUSEHOLDS` distinct households are picked from real transaction activity in `line`/`header` (not an arbitrary ID-ordered slice of `loyalty_household`, which risks zero overlap with what people actually bought). Products are *not* independently sampled or limited: `products`/`transactions` simply keep whatever products those sampled households actually bought, however many that turns out to be. This is lower-risk than sampling both households and products independently and hoping the two samples intersect -- there's nothing to fail to overlap.
 
Customer grain is the household (`HouseholdId`), since `loyalty_household` already carries aggregate spend/trip totals and a household can hold multiple cards. `loyalty_card` is aggregated up to household grain for two signals: loyalty tenure (earliest card `EnrollmentDate`) and whether any card on the household is active (`CardStatus == "A"`). `household_active` is similarly encoded from `loyalty_household.Active == "A"` rather than passed through as a raw string (Spark's `VectorAssembler` can't consume a string column, and passing it through unencoded was a bug in an earlier pass of this notebook). There's no real tier/segment field in either source table today, so the old placeholder `loyalty_tier` feature has been dropped rather than mapped to something that isn't actually a tier -- `02_propensity_train.ipynb` and friends will need a follow-up pass to remove the `loyalty_tier_idx` feature they still reference.

Products come from `item_master`, keyed by `Sku` -- assumed (not yet confirmed against real data) to match `RetailItemId` on the transaction `line` table. `category` is mapped from `ProductGroup`; swap for `CmSubcategory` / `IriCategory` / `IriSubcategory` if the business wants a different merchandising level.

Transactions are `line` joined to `header` on the natural transaction key (`TransactionDate`, `TransactionSequence`, `StoreId`, `TerminalId`, `TerminalSequence`), with `LoyaltyCardId` taken from `header` and then mapped to `HouseholdId` via `loyalty_card` (transactions key off the individual card; customers here are keyed by household). Voided, unscanned, and non-merchandise lines are filtered out.


In [5]:
# --- Determine the sample, then customers: pick households that actually have transaction
# history (not an arbitrary ID-ordered slice of loyalty_household, which could easily have zero
# overlap with what people actually bought), then build the customer profile for that sample. ---
household = spark.table(source_table("loyalty_household"))
card = spark.table(source_table("loyalty_card"))
header = spark.table(source_table("header"))
line = spark.table(source_table("line"))

TXN_KEY = ["TransactionDate", "TransactionSequence", "StoreId", "TerminalId", "TerminalSequence"]

# ASSUMPTION: RetailItemId on `line` matches item_master.Sku -- confirm once real data is queryable.
# ASSUMPTION: WasItemVoided / WasItemScanned / IsNonMerchandiseItem are boolean-like (cast handles
# either an actual boolean or a 0/1 encoding) -- adjust the filter if the real encoding differs.
line_selected = (
    line
    .filter(
        (F.col("WasItemVoided").cast("int") == 0)
        & (F.col("WasItemScanned").cast("int") == 1)
        & (F.col("IsNonMerchandiseItem").cast("int") == 0)
    )
    .select(
        *TXN_KEY,
        F.col("RetailItemId").alias("product_id"),
        F.col("NetSalesAmount").alias("amount"),
        F.col("QuantitySold").alias("quantity"),
    )
)
header_selected = header.select(*TXN_KEY, "LoyaltyCardId")
card_xwalk = card.select("LoyaltyCardId", F.col("HouseholdId").alias("customer_id"))

# every real transaction line, with customer_id/product_id resolved -- not yet sampled down.
# Cached because sample_customer_ids/sample_product_ids/transactions (below and in the next two
# cells) all derive from this, and it's expensive to recompute from line/header each time.
txn_base = (
    line_selected.join(header_selected, TXN_KEY, "inner")
    .join(card_xwalk, "LoyaltyCardId", "inner")
    .select(
        "customer_id",
        "product_id",
        F.to_date(F.col("TransactionDate")).alias("txn_date"),
        "amount",
        "quantity",
    )
    .cache()
)

# Sample households with real activity, not a blind ID-ordered slice of loyalty_household.
# Products are NOT independently sampled/limited -- keep whatever products these households
# actually bought ("let the items fall as they will"), which removes any risk of a household
# sample and a product sample failing to overlap.
sample_customer_ids = txn_base.select("customer_id").distinct().orderBy("customer_id").limit(NUM_SAMPLE_HOUSEHOLDS)
sample_product_ids = txn_base.join(F.broadcast(sample_customer_ids), "customer_id").select("product_id").distinct()

# collapse card to one row per household: earliest enrollment (loyalty tenure) + any active card
card_agg = card.groupBy("HouseholdId").agg(
    F.min("EnrollmentDate").alias("EnrollmentDate"),
    F.max(F.when(F.col("CardStatus") == "A", 1).otherwise(0)).alias("HasActiveCard"),
)

customers = (
    household.join(card_agg, "HouseholdId", "left")
    .join(F.broadcast(sample_customer_ids.withColumnRenamed("customer_id", "HouseholdId")), "HouseholdId", "inner")
    .select(
        F.col("HouseholdId").alias("customer_id"),
        F.when(F.col("Active") == "A", 1).otherwise(0).alias("household_active"),
        F.col("NumberInHousehold").alias("number_in_household"),
        F.col("RewardsBalance").alias("rewards_balance"),
        F.col("LtdTotalSales").alias("ltd_total_sales"),
        F.col("LtdTotalTrips").alias("ltd_total_trips"),
        F.col("LastShoppingDate").alias("last_shopping_date"),
        F.col("EnrollmentDate").alias("enrollment_date"),
        F.coalesce(F.col("HasActiveCard"), F.lit(0)).alias("has_active_card"),
    )
    .orderBy("customer_id")
)
customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table("customers"))
customers.show(5)

StatementMeta(, 1bdd9bd9-7b7e-488c-8793-0185123f39ec, 11, Finished, Cancelled, Cancelled, False)

Py4JJavaError: An error occurred while calling o8360.saveAsTable.
: org.apache.spark.SparkException: Job 40 cancelled part of cancelled job group 11
	at org.apache.spark.scheduler.DAGScheduler.failJobAndIndependentStages(DAGScheduler.scala:3241)
	at org.apache.spark.scheduler.DAGScheduler.handleJobCancellation(DAGScheduler.scala:3111)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleJobGroupCancelled$4(DAGScheduler.scala:1278)
	at scala.runtime.java8.JFunction1$mcVI$sp.apply(JFunction1$mcVI$sp.java:23)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:79)
	at org.apache.spark.scheduler.DAGScheduler.handleJobGroupCancelled(DAGScheduler.scala:1277)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3403)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onReceive(DAGScheduler.scala:3379)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onReceive(DAGScheduler.scala:3368)
	at org.apache.spark.util.EventLoop$$an

In [4]:
# --- Products: real Alex Lee item_master table, restricted to whatever the sampled
# households above actually bought (see sample_product_ids in the previous cell) ---
# ASSUMPTION: ProductGroup is the category grain for this model -- swap for
# CmSubcategory / IriCategory / IriSubcategory if the business wants a different level
item_master = spark.table(source_table("item_master"))

products = (
    item_master
    .join(F.broadcast(sample_product_ids.withColumnRenamed("product_id", "Sku")), "Sku", "inner")
    .select(
        F.col("Sku").alias("product_id"),
        F.col("ItemDescription").alias("item_description"),
        F.col("ProductGroup").alias("category"),
    )
    .orderBy("product_id")
)
products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table("products"))
products.show(5)

StatementMeta(, ee7a9c60-04e2-4489-be81-5acaa8e7d2c3, 6, Finished, Available, Finished, False)

+----------+------------------+--------+
|product_id|  item_description|category|
+----------+------------------+--------+
|      1440|  Regular Gasoline|    NULL|
|      1441|Mid-Grade Gasoline|    NULL|
|      1442|  Premium Gasoline|    NULL|
|      1443|       Diesel - #2|    NULL|
|      3616|   WASH ENVY APPLE|    NULL|
+----------+------------------+--------+
only showing top 5 rows



In [5]:
# --- Transactions: the real transaction lines underlying the sample above. Filtering by
# `products` too (not just `customers`) drops any transaction line whose RetailItemId has no
# matching item_master.Sku row -- a real data-quality guard, not a sampling restriction, since
# `products` was itself built from exactly what these customers bought. ---
transactions = (
    txn_base
    .join(F.broadcast(customers.select("customer_id")), "customer_id", "inner")
    .join(F.broadcast(products.select("product_id")), "product_id", "inner")
)
transactions.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table("transactions"))
transactions.show(5)
print(f"{transactions.count()} transaction rows for {customers.count()} households x {products.count()} products")

StatementMeta(, ee7a9c60-04e2-4489-be81-5acaa8e7d2c3, 7, Submitted, Running, Running, True)

## 2. Build labeled training frame

**Trending features:** `purchases_last_7d`, `purchases_last_30d`, and `purchased_last_30d_flag` below are trailing-window features -- exactly the kind of "has this customer bought something in the last N days" signal that tends to matter most for a propensity model. `AS_OF_DATE` is fixed (rather than `current_date()`) so labels are computed as of a known point in time, avoiding leakage -- switch it to `current_date()` once scoring against live data.

**Tenure-normalized features:** `ltd_total_sales` / `ltd_total_trips` are lifetime totals -- used raw, they'd confound tenure with engagement (a long-tenured household always outscores a new one regardless of how engaged either currently is). `tenure_days` (from `enrollment_date`), `avg_basket_size` (`ltd_total_sales / ltd_total_trips`), and `spend_per_tenure_day` (`ltd_total_sales / tenure_days`) turn those lifetime totals into rates instead.

Label placeholder: whether the customer purchases from the product's category again in the following period. Replace with the real forward-looking label once the modeling window is defined with the business team.

*Note:* the sample scoped in section 1 (`NUM_SAMPLE_HOUSEHOLDS`) is still small, so `02_propensity_train.ipynb`'s random train/test split can occasionally land only one label class in the test set, making `areaUnderROC` undefined (Spark returns `NaN`). That's an artifact of the sample size, not a bug -- it goes away as the sample limit is raised.


In [6]:
AS_OF_DATE = F.to_date(F.lit("2026-08-10"))  # TODO: F.current_date() once scoring live data

txns = spark.table(output_table("transactions")) \
    .join(spark.table(output_table("products")), "product_id") \
    .withColumn("days_since_txn", F.datediff(AS_OF_DATE, F.col("txn_date")))
custs = spark.table(output_table("customers"))

customer_product = (
    txns.groupBy("customer_id", "product_id", "category")
    .agg(
        F.min("days_since_txn").alias("recency_days"),
        F.count("*").alias("frequency_90d"),
        F.sum("amount").alias("monetary_90d"),
        F.sum(F.when(F.col("days_since_txn") <= 7, 1).otherwise(0)).alias("purchases_last_7d"),
        F.sum(F.when(F.col("days_since_txn") <= 30, 1).otherwise(0)).alias("purchases_last_30d"),
    )
    .withColumn("purchased_last_30d_flag", F.when(F.col("purchases_last_30d") > 0, 1).otherwise(0))
    .withColumn("seasonal_index", F.month(AS_OF_DATE))
    .join(custs, "customer_id")
    # Households with no card record (left join in the customers table above) or nulls in the
    # raw loyalty_household totals would otherwise carry nulls all the way into the feature
    # vector -- VectorAssembler's handleInvalid="skip" then silently drops those rows, and with
    # a small sample that can drop the *entire* training set. Coalesce to 0 so every row survives.
    .withColumn("number_in_household", F.coalesce(F.col("number_in_household"), F.lit(0)))
    .withColumn("rewards_balance", F.coalesce(F.col("rewards_balance"), F.lit(0.0)))
    .withColumn("ltd_total_sales", F.coalesce(F.col("ltd_total_sales"), F.lit(0.0)))
    .withColumn("ltd_total_trips", F.coalesce(F.col("ltd_total_trips"), F.lit(0.0)))
    # tenure/rate features: lifetime totals alone confound tenure with engagement (a 10-year
    # household will always outscore a 1-month household on raw LTD totals) -- normalize instead
    .withColumn("tenure_days", F.coalesce(F.datediff(AS_OF_DATE, F.col("enrollment_date")), F.lit(0)))
    .withColumn(
        "avg_basket_size",
        F.when(F.col("ltd_total_trips") > 0, F.col("ltd_total_sales") / F.col("ltd_total_trips")).otherwise(F.lit(0.0)),
    )
    .withColumn(
        "spend_per_tenure_day",
        F.when(F.col("tenure_days") > 0, F.col("ltd_total_sales") / F.col("tenure_days")).otherwise(F.lit(0.0)),
    )
    # placeholder label: recent, frequent buyers are treated as repeat purchasers
    .withColumn(
        "will_purchase_next_period",
        F.when((F.col("recency_days") < 45) & (F.col("frequency_90d") >= 2), 1).otherwise(0),
    )
    .withColumn("created_at", F.current_timestamp())
)

customer_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table("training_frame"))
customer_product.show(5)

StatementMeta(, 1bdd9bd9-7b7e-488c-8793-0185123f39ec, 13, Finished, Available, Finished, False)

+-----------+-----------+--------------------+------------+-------------+------------+-----------------+------------------+-----------------------+--------------+----------------+-------------------+---------------+---------------+---------------+------------------+---------------+---------------+-----------+------------------+--------------------+-------------------------+--------------------+
|customer_id| product_id|            category|recency_days|frequency_90d|monetary_90d|purchases_last_7d|purchases_last_30d|purchased_last_30d_flag|seasonal_index|household_active|number_in_household|rewards_balance|ltd_total_sales|ltd_total_trips|last_shopping_date|enrollment_date|has_active_card|tenure_days|   avg_basket_size|spend_per_tenure_day|will_purchase_next_period|          created_at|
+-----------+-----------+--------------------+------------+-------------+------------+-----------------+------------------+-----------------------+--------------+----------------+-------------------+-----

Next: `02_propensity_train.ipynb` for a simple baseline train/test pass, then `04_propensity_train_advanced.ipynb` for hyperparameter tuning.

In [1]:
df = spark.sql("SELECT * FROM inviso_sandbox.dbo.product_purchase_propensity_training_frame")
print(df.count())
display(df)

StatementMeta(, 0df13a56-175e-4f05-8b87-f6ba5c1ff6d4, 3, Finished, Available, Finished, False)

1136


SynapseWidget(Synapse.DataFrame, d93c75f8-d61e-4125-b82f-31c95459783d)